# 1 · LangChain — the foundation
Model-agnostic interface, LCEL chains, and tool-calling.

**Run cells top to bottom.**

In [14]:
# Bootstrap: make the repo root importable so `import config` works from notebooks/
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from config import assert_key
assert_key()
print("Gateway ready.")

Gateway ready.


In [15]:
from config import get_langchain_llm
llm = get_langchain_llm()

### LCEL: `prompt | model | parser` composes like a pipe

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant for AI engineers."),
    ("human", "Explain {topic} in exactly two sentences."),
])
chain = prompt | llm | StrOutputParser()
print(chain.invoke({"topic": "the difference between an LLM and an agent"}))

An LLM (Large Language Model) is a neural network trained to predict and generate text based on patterns in data, functioning as a stateless text processor. An agent is a system that uses an LLM as its reasoning engine but adds the ability to take actions, use tools, maintain memory, and pursue goals autonomously over multiple steps.


In [17]:
from config import get_langchain_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_langchain_llm()

# prompt | model | parser  — that's LCEL
chain = ChatPromptTemplate.from_template("Explain {topic} in one sentence.") | llm | StrOutputParser()

print(chain.invoke({"topic": "nomitha"}))

I don't have any information about "nomitha" - it could be a name, term, or concept from a specific field or language that I'm not familiar with.


### Tool calling — the primitive every agent framework builds on

In [18]:
from langchain_core.tools import tool

@tool
def get_stock_level(sku: str) -> int:
    'Return the number of units in stock for a given product SKU.'
    return {"scout": 12, "hauler": 3, "sentinel": 0}.get(sku.lower(), 0)

llm_with_tools = llm.bind_tools([get_stock_level])
msg = llm_with_tools.invoke("How many Hauler units are in stock?")
print(msg.tool_calls)

[{'name': 'get_stock_level', 'args': {'sku': 'Hauler'}, 'id': 'toolu_bdrk_011rzJvyZVRnprpxC34Lg51y', 'type': 'tool_call'}]


In [19]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from config import get_langchain_llm

llm = get_langchain_llm()

@tool
def inventory(sku: str) -> int:
    'Units in stock for a product SKU (scout/hauler/sentinel).'
    return {"scout": 12, "hauler": 3, "sentinel": 0}.get(sku.lower(), 0)

@tool
def price(sku: str) -> int:
    'List price in USD for a product SKU.'
    return {"scout": 18000, "hauler": 42000, "sentinel": 30000}.get(sku.lower(), 0)

tools = {"inventory": inventory, "price": price}
llm_with_tools = llm.bind_tools(list(tools.values()))

# 1) One question that needs BOTH tools -> model returns MULTIPLE tool calls
question = "What's the stock level AND unit price of the Hauler?"
ai_msg = llm_with_tools.invoke(question)

print("Tool calls the model requested:")
for tc in ai_msg.tool_calls:
    print("  -", tc["name"], tc["args"])

# 2) Execute each requested call -> each returns a ToolMessage (matched by id)
messages = [HumanMessage(question), ai_msg]
for tc in ai_msg.tool_calls:
    tool_msg = tools[tc["name"]].invoke(tc)      # pass the whole call dict incl. 'id'
    print("   ran", tc["name"], "->", tool_msg.content)
    messages.append(tool_msg)

# 3) Send the results back -> model writes the final answer
final = llm_with_tools.invoke(messages)
print("\nFINAL:", final.content)

Tool calls the model requested:
  - inventory {'sku': 'hauler'}
  - price {'sku': 'hauler'}
   ran inventory -> 3
   ran price -> 42000

FINAL: The Hauler has:
- **Stock level**: 3 units in stock
- **Unit price**: $42,000 USD


In [20]:
print(get_stock_level.invoke(msg.tool_calls[0]["args"]))

3


### Streaming

In [21]:
for chunk in chain.stream({"topic": "why observability matters for agents"}):
    print(chunk, end="", flush=True)

Observability matters for agents because without visibility into their reasoning process, tool usage, and decision-making steps, you cannot debug failures, improve performance, or trust them in production environments.

In [22]:
# your code here